### Using library for metacells-2 directly 


#### using version metacells==0.9.5

In [2]:
import metacells as mc


/opt/anaconda3/envs/octopus/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import numpy as np
import anndata as ad
import scanpy as sc

In [4]:

final_snn= np.load('/Users/anushka/Undergraduate-Project/snn_matrices/final_snn.npy')

In [5]:
adata = sc.read_h5ad("/Users/anushka/Undergraduate-Project/updated_dataset/st_data.h5ad")  # Load from file

adata = sc.read_h5ad("/Users/anushka/Undergraduate-Project/updated_dataset/sc_data.h5ad.1")  


In [7]:
final_snn.shape

(25932, 25932)

In [6]:
adata.obsp["snn"] = final_snn  # Store the computed SNN matrix


ValueError: Value passed for key 'snn' is of incorrect shape. Values of obsp must match dimensions ('obs', 'obs') of parent. Value had shape (25932, 25932) while it should have had (7416, 7416).

In [30]:
print(adata.obsp["connectivities"].shape)
num_neighbors = np.array((adata.obsp["connectivities"] > 0).sum(axis=1)).flatten()
print(f"Mean neighbors: {num_neighbors.mean()}, Median: {np.median(num_neighbors)}, Min: {num_neighbors.min()}, Max: {num_neighbors.max()}")


(25932, 25932)
Mean neighbors: 4.830711090544501, Median: 3.0, Min: 0, Max: 46


####  Mean neighbors: ~4.83 → Cells are relatively sparsely connected.
#### Median neighbors: 3 → Many cells have very few connections.
#### Max neighbors: 46 → Some dense clusters exist, but overall, connections are limited.


In [36]:
print(adata.X.dtype)


float64


In [35]:
mc.pipeline.compute_direct_metacells(
    adata,
    what="__x__",  # Uses the main expression matrix
    target_metacell_size=40,  # Adjusted based on your connectivity
    min_metacell_size=10,  # Ensure small but stable metacells
    knn_k=15,
    random_seed=42   # Aligns with your original SNN k-value 
)


AssertionError: 

In [28]:
print(adata.obsp.keys())

KeysView(PairwiseArrays with keys: connectivities)


### Step by step pipeline

In [32]:
help(mc.pipeline.compute_direct_metacells)


Help on function compute_direct_metacells in module metacells.pipeline.direct:

compute_direct_metacells(adata: anndata._core.anndata.AnnData, what: Union[str, numpy.ndarray, metacells.utilities.CompressedMatrix, pandas.core.frame.DataFrame, metacells.utilities.SparseMatrix] = '__x__', *, select_downsample_min_samples: int = 750, select_downsample_min_cell_quantile: float = 0.05, select_downsample_max_cell_quantile: float = 0.5, select_min_gene_total: Optional[int] = 50, select_min_gene_top3: Optional[int] = 4, select_min_gene_relative_variance: Optional[float] = 0.1, select_min_genes: int = 100, cells_similarity_value_regularization: float = 0.14285714285714285, cells_similarity_log_data: bool = True, cells_similarity_method: str = 'abs_pearson', target_metacell_size: int = 48, min_metacell_size: int = 12, target_metacell_umis: int = 160000, cell_umis: Optional[numpy.ndarray] = None, knn_k: Optional[int] = None, knn_k_size_factor: float = 2, knn_k_umis_quantile: float = 0.1, min_knn_k

In [16]:
sc.pp.highly_variable_genes(adata, n_top_genes=1000)

In [17]:
selected_genes = adata.var_names[adata.var['highly_variable']].tolist()
mc.pipeline.mark_select_genes(adata, select_gene_names=selected_genes)

set unnamed.var[select_gene]: 1003 true (3.868%) out of 25932 bools


In [ ]:
# Perform clustering
mc.pl.find_knn_graph(adata)  # Compute nearest-neighbor graph
mc.pl.find_metacells(adata, random_seed=42)  # Cluster into metacells

# Extract MetaCells clustering labels
metacell_labels = adata.obs["metacell"]

In [10]:
print(dir(mc))


['SHOULD_CHECK_AVX2', '__author__', '__builtins__', '__cached__', '__doc__', '__email__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'extensions', 'parameters', 'pipeline', 'pl', 'should_check_avx2', 'sys', 'tl', 'tools', 'ut', 'utilities']


In [12]:
import metacells.pipeline as pl

print(dir(pl))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'collect', 'collect_metacells', 'compute_direct_metacells', 'compute_divide_and_conquer_metacells', 'compute_for_mcview', 'compute_knn_by_markers', 'compute_target_pile_size', 'compute_umap_by_markers', 'direct', 'divide_and_conquer', 'divide_and_conquer_pipeline', 'exclude', 'exclude_cells', 'exclude_genes', 'extract_clean_data', 'extract_selected_data', 'get_max_parallel_piles', 'guess_max_parallel_piles', 'mark', 'mark_essential_genes', 'mark_ignored_genes', 'mark_lateral_genes', 'mark_noisy_genes', 'mark_select_genes', 'mcview', 'outliers_projection_pipeline', 'projection', 'projection_pipeline', 'relate_to_lateral_genes', 'related_genes', 'select', 'set_max_parallel_piles', 'umap', 'write_projection_weights']


In [25]:
# Step 1: Mark marker genes
mc.tl.find_named_genes(adata)  # Automatically detect markers

# Step 2: Compute KNN graph using marker genes
mc.pipeline.compute_knn_by_markers(adata, k=15, reproducible=True)


KeyError: 'unknown v data name: marker_gene'

In [22]:
# Compute KNN graph using selected marker genes
mc.pipeline.compute_knn_by_markers(adata, k=15, reproducible=True)

# Compute UMAP for visualization
mc.pipeline.compute_umap_by_markers(adata)


KeyError: 'unknown v data name: marker_gene'

### Kaggle implementation ( to edit)

In [8]:
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
import umap


In [ ]:
import umap
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the original scRNA-seq and MetaCells data
sc_data = np.load("/Users/anushka/Undergraduate-Project/updated_dataset/sc_data.h5ad.1")  # Original single-cell data
st_data = np.load("/Users/anushka/Undergraduate-Project/updated_dataset/st_data.h5ad")  # Original spatial transcriptomics data
metacell_data = np.load("kaggle/working/metacell_data.npy")  # Computed MetaCells

# Concatenate all data for joint UMAP embedding
combined_data = np.vstack([sc_data, st_data, metacell_data])

# UMAP projection
umap_model = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
umap_embedding = umap_model.fit_transform(combined_data)

# Split UMAP embeddings
num_sc = sc_data.shape[0]
num_st = st_data.shape[0]

umap_sc = umap_embedding[:num_sc]
umap_st = umap_embedding[num_sc:num_sc + num_st]
umap_metacell = umap_embedding[num_sc + num_st:]
